In [4]:
import math, heapq
G = {
    's': [('t', 6), ('v', 7)],
    't': [('k', 4)],
    'k': [('h', 4), ('v', 2)],
    'v': [('k', 2), ('w', 4), ('p', 3)],
    'w': [('h', 1)],
    'h': [('p', 3)],
    'p': [('h', 3)],
}

def dijkstra(graph, source):
    dist = {v: math.inf for v in graph}
    prev = {}
    dist[source] = 0
    pq = [(0, source)]
    while pq:
        d, u = heapq.heappop(pq)
        if d != dist[u]:
            continue
        for v, w in graph[u]:
            nd = d + w
            if nd < dist[v]:
                dist[v] = nd
                prev[v] = u
                heapq.heappush(pq, (nd, v))
    return dist, prev

def path(prev, dest):
    if dest not in prev and dest != 's':
        return None
    seq = [dest]
    while seq[-1] in prev:
        seq.append(prev[seq[-1]])
    return list(reversed(seq))

dist, prev = dijkstra(G, 's')
for v in G:
    print(f"{v}: {dist[v]:2}  مسیر: {path(prev, v)}")


s:  0  مسیر: ['s']
t:  6  مسیر: ['s', 't']
k:  9  مسیر: ['s', 'v', 'k']
v:  7  مسیر: ['s', 'v']
w: 11  مسیر: ['s', 'v', 'w']
h: 12  مسیر: ['s', 'v', 'w', 'h']
p: 10  مسیر: ['s', 'v', 'p']


In [5]:
# pip install pulp
import pulp as pl

def shortest_path_lp(graph, s, d):
    edges = [(u, v, w) for u in graph for v, w in graph[u]]

    model = pl.LpProblem("shortest_path", pl.LpMinimize)

    x = {(u, v): pl.LpVariable(f"x_{u}_{v}", cat="Binary")
         for u, v, _ in edges}

    model += pl.lpSum(w * x[(u, v)] for u, v, w in edges)

    for node in graph:
        inflow  = pl.lpSum(x[(u, v)] for u, v, _ in edges if v == node)

        outflow = pl.lpSum(x[(u, v)] for u, v, _ in edges if u == node)

        rhs = 0
        if node == s: rhs = 1      
        if node == d: rhs = -1    

        model += outflow - inflow == rhs

    model.solve(pl.PULP_CBC_CMD(msg=False))

    total_cost = pl.value(model.objective)

    succ = {u: v for (u, v) in x if pl.value(x[(u, v)]) > 0.5}
    path = [s]
    while path[-1] != d:
        path.append(succ[path[-1]])

    return total_cost, path


G = {
    's': [('t', 6), ('v', 7)],
    't': [('k', 4)],
    'k': [('h', 4), ('v', 2)],
    'v': [('k', 2), ('w', 4), ('p', 3)],
    'w': [('h', 1)],
    'h': [('p', 3)],
    'p': [('h', 3)],
}

for dest in ['t', 'v', 'k', 'w', 'p', 'h']:
    cost, path = shortest_path_lp(G, 's', dest)
    print(f"s → {dest:<2}:  مسیر = {path},  طول = {cost}")


s → t :  مسیر = ['s', 't'],  طول = 6.0
s → v :  مسیر = ['s', 'v'],  طول = 7.0
s → k :  مسیر = ['s', 'v', 'k'],  طول = 9.0
s → w :  مسیر = ['s', 'v', 'w'],  طول = 11.0
s → p :  مسیر = ['s', 'v', 'p'],  طول = 10.0
s → h :  مسیر = ['s', 'v', 'w', 'h'],  طول = 12.0
